# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [4]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)
df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())


Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [5]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [6]:
# Task 1
continent_choice = "Asia"
task1_df = df.loc[
    (df["continent"] == continent_choice) & (df["year"].isin([2002, 2007]))
].copy()
task1_df["year"] = task1_df["year"].astype(str)

median_points = (
    task1_df.groupby("year", as_index=False)[["gdpPercap", "lifeExp"]]
    .median()
    .sort_values("year")
)

fig = px.scatter(
    task1_df,
    x="gdpPercap",
    y="lifeExp",
    color="year",
    hover_name="country",
    log_x=True,
    color_discrete_map={"2002": "#9e9e9e", "2007": "#1f77b4"},
    title="Asia moved up and to the right from 2002 to 2007",
    labels={
        "gdpPercap": "GDP per capita (log scale)",
        "lifeExp": "Life expectancy",
        "year": "Year",
    },
)

fig.add_annotation(
    x=median_points.iloc[1]["gdpPercap"],
    y=median_points.iloc[1]["lifeExp"],
    ax=median_points.iloc[0]["gdpPercap"],
    ay=median_points.iloc[0]["lifeExp"],
    xref="x",
    yref="y",
    axref="x",
    ayref="y",
    text="Median country position improved by 2007",
    showarrow=True,
    arrowhead=3,
    arrowcolor="#1f77b4",
)

fig.update_traces(marker=dict(size=11, opacity=0.8, line=dict(width=0.5, color="white")))
fig.update_layout(template="plotly_white")
fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [7]:
# Task 2
highlight_continent = "Asia"
task2_df = df.loc[df["year"] == 2007].copy()
task2_df["highlight"] = task2_df["continent"].where(
    task2_df["continent"] == highlight_continent,
    "Other continents",
)

fig = px.scatter(
    task2_df,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="highlight",
    hover_name="country",
    log_x=True,
    size_max=55,
    category_orders={"highlight": [highlight_continent, "Other continents"]},
    color_discrete_map={highlight_continent: "#1f77b4", "Other continents": "#c7c7c7"},
    title="In 2007, Asia combined huge populations with a very wide development range",
    labels={
        "gdpPercap": "GDP per capita (log scale)",
        "lifeExp": "Life expectancy",
        "highlight": "Highlight",
        "pop": "Population",
    },
)

china = task2_df.loc[task2_df["country"] == "China"].iloc[0]
fig.add_annotation(
    x=china["gdpPercap"],
    y=china["lifeExp"],
    text="Asia stands out because several of the world's largest populations cluster here",
    showarrow=True,
    arrowhead=3,
    ax=80,
    ay=-60,
    bgcolor="white",
)

fig.update_traces(marker=dict(opacity=0.75, line=dict(width=0.5, color="white")))
fig.update_layout(template="plotly_white")
fig.show()
